# Autofit tool for aligning data with the modes

`pyMMF.tools.autofit` allows for finding the transformation to center the fiber axis and the zoom to fit the simulated modes.

In [1]:
import matplotlib.pyplot as plt
import numpy as np



import os, sys
print(os.path.abspath('../..'))
os.chdir(os.path.abspath('../..'))
# sys.path.insert(0, os.path.abspath('..'))


import pyMMF
print(pyMMF.__path__)
from pyMMF.functions import colorize
from pyMMF.tools.autofit import Autofit

/home/spopoff/mycore/dev/pyMMF
['/home/spopoff/mycore/dev/pyMMF/pyMMF']


## 1. Fiber parameters

In [2]:
NA = 0.2
radius = 25 # in microns
areaSize = 3.*radius # calculate the field on an area larger than the diameter of the fiber
n_points_modes = 48 # resolution of the window2
n1 = 1.45 # index of refraction at r=0 (maximum)
wl = 1.55 # wavelength in microns
curvature = None
k0 = 2.*np.pi/wl

r_max = 3.2*radius
npoints_search = 2**8
dh = 2*radius/npoints_search

# solver parameters
solver_options = {
    'degenerate_mode': 'exp', # 'eig' or 'prop'
    'min_radius_bc': 1.5,   # min large radial boundary condition
    'N_beta_coarse': 1_000, # number of steps of the initial coarse scan
    'change_bc_radius_step': 0.95, #change of the large radial boundary condition if fails 
    'dh': dh,               # radial resolution during the computation
    'r_max': r_max,         # max radius to calculate (and first try for large radial boundary condition)
}

## 2. Compute the mode with radial solver

In [3]:
profile = pyMMF.IndexProfile(
    npoints = n_points_modes, 
    areaSize = areaSize
)
profile.initParabolicGRIN(n1=n1, a=radius, NA=NA)

solver = pyMMF.propagationModeSolver()
solver.setIndexProfile(profile)
solver.setWL(wl)


modes = solver.solve(solver='radial',
                    curvature = curvature,
                    options = solver_options
                    )


2026-04-22 11:52:54,049 - pyMMF.core [DEBUG  ]  Debug mode ON.
2026-04-22 11:52:54,052 - pyMMF.solv [INFO   ]  Searching for modes with beta_min=5.821637357564584, beta_max=5.877818513168
2026-04-22 11:52:54,079 - pyMMF.solv [INFO   ]  Found 5 radial mode(s) for m=0
2026-04-22 11:52:54,080 - pyMMF.solv [INFO   ]  Searching propagation constant for |l| = 1
2026-04-22 11:52:54,082 - pyMMF.solv [ERROR  ]  Field limit 1.0 at the founded beta=0.05066217542815342 is greater than field_limit_tol=0.001
2026-04-22 11:52:54,083 - pyMMF.solv [WARNING]  Boundary condition could not be met.
2026-04-22 11:52:54,084 - pyMMF.solv [WARNING]  Retrying by changing r_max to 3.04a
2026-04-22 11:52:54,086 - pyMMF.solv [ERROR  ]  Field limit 1.0 at the founded beta=0.05066217542815342 is greater than field_limit_tol=0.001
2026-04-22 11:52:54,086 - pyMMF.solv [WARNING]  Boundary condition could not be met.
2026-04-22 11:52:54,087 - pyMMF.solv [WARNING]  Retrying by changing r_max to 2.89a
2026-04-22 11:52:54,

## 3. Simulating the shift/zoom of the data by applying a transformation of the modes

In [ ]:
from scipy.ndimage import zoom as nd_zoom

# Resample the modes onto distinct output and input grids, to emulate the
# typical experimental case where the camera (output) and SLM (input)
# sample the field at different resolutions.
#   M0_out has shape (n_out**2, n_modes)
#   M0_in  has shape (n_in**2,  n_modes)
# The resulting TM_0 is therefore non-square: (n_out**2, n_in**2).
n_out = 48
n_in = 32

M0_hr = modes.getModeMatrix()
N_hr = modes.indexProfile.npoints
n_modes = M0_hr.shape[1]

def _resample_modes(M_hr, target_N):
    factor = target_N / N_hr
    out = np.empty((target_N * target_N, M_hr.shape[1]), dtype=M_hr.dtype)
    for k in range(M_hr.shape[1]):
        img = M_hr[:, k].reshape(N_hr, N_hr)
        real = nd_zoom(img.real, factor, order=3)
        imag = nd_zoom(img.imag, factor, order=3)
        out[:, k] = (real + 1j * imag).ravel()
    return out

M0_out = _resample_modes(M0_hr, n_out)
M0_in = _resample_modes(M0_hr, n_in)

af = Autofit(modes)

TM_mode_basis = modes.getPropagationMatrix(1e4)  # propagation of 1 cm

# Non-square TM: (n_out**2, n_in**2)
TM_0 = M0_out @ TM_mode_basis @ M0_in.conj().T

# Asymmetric misalignment: different zoom and shift on output and input sides.
# params = [zoom, shift_out, shift_in] with zoom = (s_out, s_in).
params = [(1.15, 0.9), (2., 3.), (-4., 3.)]
TM_misaligned = af.transform(TM_0, params=params)

In [ ]:
# compute average output intensity
mean_I0 = np.mean(np.abs(TM_0)**2, axis=1).reshape((n_out, n_out))
mean_I1 = np.mean(np.abs(TM_misaligned)**2, axis=1).reshape((n_out, n_out))

plt.figure(figsize=(12,7))
plt.subplot(1,2,1)
plt.imshow(mean_I0, cmap='hot', interpolation='nearest')
plt.axis('off')
plt.title('Mean Intensity of computed modes')
plt.subplot(1,2,2)
plt.imshow(mean_I1, cmap='hot', interpolation='nearest')
plt.axis('off')
plt.title('Mean Intensity of simulated data')
plt.show()

## 4. Autofit to find the transformation to recenter/rezoom the data

`realign_TM` splits the alignment between the TM and the modes:

- **The TM is only recentered** — pure translations are applied on the output and input sides so that the mean intensity is centered on the grid. **No zoom is applied to the TM.**
- **The zoom is applied to the modes.** The scale factors are estimated from the TM (from the RMS size of `|TM|²` compared to the modes intensity, then optionally fine-tuned by correlation) and used to resample the high-resolution modes onto the output and input TM grids. The returned `new_modes_out_af` / `new_modes_in_af` are the resampled, globally-normalized mode matrices.

In [ ]:
af = Autofit(modes)
TM_realigned, new_modes_out_af, new_modes_in_af = af.realign_TM(
    TM_misaligned,
    params={'threshold': 0.5},
)

# Output-side: average along inputs (axis=1), reshape to (n_out, n_out)
I_mean_tm_out_af = (np.abs(TM_realigned) ** 2).sum(axis=1).reshape([n_out] * 2)
I_modes_out_af = (np.abs(new_modes_out_af) ** 2).sum(axis=1).reshape([n_out] * 2)

# Input-side: average along outputs (axis=0), reshape to (n_in, n_in)
I_mean_tm_in_af = (np.abs(TM_realigned) ** 2).sum(axis=0).reshape([n_in] * 2)
I_modes_in_af = (np.abs(new_modes_in_af) ** 2).sum(axis=1).reshape([n_in] * 2)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(I_mean_tm_out_af, cmap='hot')
axes[0, 0].set_title('TM mean I (output)')
axes[0, 0].axis('off')
axes[0, 1].imshow(I_modes_out_af, cmap='hot')
axes[0, 1].set_title('Resampled modes mean I (output)')
axes[0, 1].axis('off')
axes[1, 0].imshow(I_mean_tm_in_af, cmap='hot')
axes[1, 0].set_title('TM mean I (input)')
axes[1, 0].axis('off')
axes[1, 1].imshow(I_modes_in_af, cmap='hot')
axes[1, 1].set_title('Resampled modes mean I (input)')
axes[1, 1].axis('off')
plt.tight_layout()
plt.show()

# Projection of the realigned TM onto the resampled mode basis.
# realign_TM returns modes with a global Frobenius normalization; for the
# conversion-efficiency metric we re-normalize each mode column so the
# basis is (approximately) orthonormal.
new_modes_out_col = new_modes_out_af / np.linalg.norm(
    new_modes_out_af, axis=0, keepdims=True
)
new_modes_in_col = new_modes_in_af / np.linalg.norm(
    new_modes_in_af, axis=0, keepdims=True
)
new_TM_modes_af = new_modes_out_col.conj().T @ TM_realigned @ new_modes_in_col
print(
    f"Energy ratio after projection: "
    f"{np.linalg.norm(new_TM_modes_af) / np.linalg.norm(TM_realigned):.6f}"
)

Computation effiency of the projection of the TM onto the mode basis

We compute the ratio of the energy of the TM after projection in the mode basis with the one of the initial TM.
If we have the right modes, and the zoom/shift is compensated for, it should be equal to 1.

In [ ]:
conversion_efficiency_before =  np.linalg.norm(M0.conj().T @ TM_misaligned @ M0)/np.linalg.norm(TM_misaligned)
conversion_efficiency_after =  np.linalg.norm(M0.conj().T @ TM_realigned @ M0)/np.linalg.norm(TM_realigned)

print(f"Conversion efficiency before: {conversion_efficiency_before:.6f}, after: {conversion_efficiency_after:.6f}")

Conversion efficiency before: 0.811440, after: 0.997533
